In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

from implied_volatility import implied_volatility
from black_scholes import black_scholes_price

In [ ]:
S0, K0, T0, r0 = 100, 105, 0.5, 0.05
rows = []
for true_sigma in [0.10, 0.25, 0.50, 1.00]:
    for opt in ["call", "put"]:
        px = black_scholes_price(S0, K0, T0, r0, true_sigma, opt)
        recovered = implied_volatility(px, S0, K0, T0, r0, opt)
        rows.append({"type": opt, "true vol": true_sigma, "price": px,
                     "recovered vol": recovered, "abs error": abs(recovered - true_sigma)})
pd.DataFrame(rows)

In [ ]:
TICKER = "SPY"
R = 0.04                      # SET THIS to the current short-term Treasury yield
MIN_T, MAX_T = 0.04, 1.0      # keep expiries between ~2 weeks and 1 year
MONEYNESS_RANGE = (0.8, 1.2)  # keep strikes within 80%-120% of spot
MIN_OPEN_INTEREST = 10

tk = yf.Ticker(TICKER)
spot = float(tk.history(period="5d")["Close"].dropna().iloc[-1])
today = pd.Timestamp.today().normalize()

frames = []
for exp in tk.options:
    ttm = (pd.Timestamp(exp) - today).days / 365
    if not (MIN_T <= ttm <= MAX_T):
        continue
    chain = tk.option_chain(exp)
    for opt_type, side in [("call", chain.calls), ("put", chain.puts)]:
        side = side.copy()
        side["option_type"] = opt_type
        side["expiry"] = exp
        side["ttm"] = ttm
        frames.append(side)

raw = pd.concat(frames, ignore_index=True)
print(f"Spot: {spot:.2f} | raw contracts: {len(raw)} | expiries: {raw['expiry'].nunique()}")

In [ ]:
df = raw.copy()
df["moneyness"] = df["strike"] / spot

# mid price when live quotes exist, otherwise last traded price (e.g. weekends)
has_quote = (df["bid"] > 0) & (df["ask"] > 0)
df["market_price"] = np.where(has_quote, (df["bid"] + df["ask"]) / 2, df["lastPrice"])

# out-of-the-money options only: puts below spot, calls at or above spot
otm = ((df["option_type"] == "put") & (df["strike"] < spot)) | \
      ((df["option_type"] == "call") & (df["strike"] >= spot))

df = df[otm
        & df["moneyness"].between(*MONEYNESS_RANGE)
        & (df["market_price"] > 0)
        & (df["openInterest"].fillna(0) >= MIN_OPEN_INTEREST)]

df["implied_vol"] = [
    implied_volatility(p, spot, k, t, R, o)
    for p, k, t, o in zip(df["market_price"], df["strike"], df["ttm"], df["option_type"])
]
df = df.dropna(subset=["implied_vol"])
df = df[df["implied_vol"].between(0.02, 1.5)]
print(f"Contracts used: {len(df)}")
df[["expiry", "option_type", "strike", "market_price", "implied_vol"]].head(10)

In [ ]:
expiries = sorted(df["expiry"].unique())
chosen = expiries[::max(1, len(expiries) // 5)][:5]

plt.figure(figsize=(9, 5))
for exp in chosen:
    sub = df[df["expiry"] == exp].sort_values("moneyness")
    plt.plot(sub["moneyness"], sub["implied_vol"], marker="o", markersize=3,
             label=f"{exp} (T={sub['ttm'].iloc[0]:.2f}y)")
plt.axvline(1.0, color="grey", linestyle="--", linewidth=1)
plt.xlabel("Moneyness (K / S)")
plt.ylabel("Implied volatility")
plt.title(f"{TICKER} implied volatility smile")
plt.legend()
plt.show()

In [ ]:
fig = plt.figure(figsize=(11, 7))
ax = fig.add_subplot(111, projection="3d")
surf = ax.plot_trisurf(df["moneyness"], df["ttm"], df["implied_vol"],
                       cmap="viridis", linewidth=0.1, antialiased=True)
ax.set_xlabel("Moneyness (K / S)")
ax.set_ylabel("Time to expiry (years)")
ax.set_zlabel("Implied volatility")
ax.set_title(f"{TICKER} implied volatility surface")
fig.colorbar(surf, shrink=0.5, aspect=10)
plt.show()